# 서울시 무더위쉼터 후보시설 구축

서울 25개 자치구의 신규 무더위쉼터 후보를 만든다. 금천구 분석에서 쓴 절차를 서울 전체로 넓힌 것이며, 결과 파일의 열 구성도 `금천구_후보시설.csv`와 같다.

## 자료

| 자료 | 건수 | 좌표 |
|---|---|---|
| 서울시 경로당 | 3,644 | 없음 |
| 읍면동 하부행정기관 (주민센터) | 서울 427 | 없음 |
| 전국도서관 표준데이터 | 서울 536 | 있음 |
| 전국 지역보건의료기관 (보건소) | 서울 | 없음 |
| 서울시 무더위쉼터 | 4,092 | 있음. 기존 시설이자 좌표 공급원 |

복지관, 체육시설, 은행은 이번 단계에서 제외한다. 복지관과 체육시설은 전국 표준데이터가 오픈API로만 제공되고, 은행은 전국 단위 공개 자료가 없다. 서울 무더위쉼터에서 세 유형이 차지하는 비중은 10.3%다.

## 좌표를 얻는 순서

먼저 손에 있는 자료로 채우고, 남은 것만 외부 지오코딩을 쓴다. 호출 결과는 캐시에 저장하므로 다시 실행해도 재요청하지 않는다.

1. 원자료에 좌표가 있으면 그대로 쓴다 (도서관)
2. 무더위쉼터 자료와 도로명주소가 같으면 그 좌표를 쓴다
3. 무더위쉼터 자료와 지번주소가 같으면 그 좌표를 쓴다
4. 같은 자치구 안에서 시설명이 같으면 그 좌표를 쓴다
5. 남은 것은 ArcGIS로 지오코딩하고, 실패하면 OSM Nominatim으로 한 번 더 시도한다

추정 좌표는 넣지 않는다. 주소 지점이 정확히 일치할 때만 받고, 못 찾으면 빈칸으로 두고 몇 건인지 센다.

## 산출물

| 파일 | 내용 |
|---|---|
| `서울시_후보시설.csv` | 후보 전체. 같은 주소는 한 행으로 합침 |
| `서울시_후보시설_교집합.csv` | 후보 중 이미 무더위쉼터로 지정된 곳 |
| `서울시_후보시설_기존쉼터제외.csv` | 교집합을 뺀 신규 후보 |

## 1. 설정

In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

ROOT = Path.cwd().parent
RAW = ROOT / "2_데이터" / "원본"
PROC = ROOT / "2_데이터" / "가공"
PROC.mkdir(parents=True, exist_ok=True)

SIDO = "서울특별시"


def find(*names, required=True):
    """원본 폴더와 저장소 최상위를 모두 뒤진다. 없으면 None."""
    for n in names:
        for base in (RAW, ROOT):
            p = base / n
            if p.exists():
                return p
    if required:
        raise FileNotFoundError(names)
    return None


P_SHELTER = find("서울시_무더위쉼터.csv")
P_GYEONGRO = find("서울시_경로당.csv", "서울시 경로당 정보.csv")
P_DONG = find("읍면동_하부행정기관.csv", "행정안전부_읍면동 하부행정기관 현황_20251231.csv")
P_LIB = find("전국도서관.csv", "전국도서관표준데이터.csv")
P_HEALTH = find("전국_지역보건의료기관.csv", "보건복지부_전국 지역보건의료기관 현황_20251231.csv",
                required=False)
CACHE = PROC / "_geocode_seoul.json"

for p in (P_SHELTER, P_GYEONGRO, P_DONG, P_LIB, P_HEALTH):
    print("찾음:", p.name if p else "(보건소 파일 없음. 건너뜀)")

찾음: 서울시_무더위쉼터.csv
찾음: 서울시 경로당 정보.csv
찾음: 행정안전부_읍면동 하부행정기관 현황_20251231.csv
찾음: 전국도서관표준데이터.csv
찾음: 보건복지부_전국 지역보건의료기관 현황_20251231.csv


## 2. 주소 처리 함수

금천구 노트북의 함수를 서울 전체로 넓혔다. 도로명은 자치구마다 같은 이름이 있으므로(중앙로, 도서관길 등) 자치구를 키에 반드시 포함한다.

In [2]:
GU_RE = re.compile(r"(\S+구)")
DONG_RE = re.compile(r"\(([^)]*동[0-9가-힣]*)")   # 도로명주소 괄호 안의 법정동


def gu_of(text):
    """주소 문자열에서 자치구를 뽑는다."""
    t = str(text or "")
    m = GU_RE.search(t.replace(SIDO, " "))
    return m.group(1) if m else ""


def dong_of(*texts):
    """도로명주소 괄호 또는 지번주소에서 법정동을 뽑는다."""
    for t in texts:
        t = str(t or "")
        m = DONG_RE.search(t)
        if m:
            return m.group(1).split(",")[0].strip()
        m = re.search(r"\S+구\s+(\S+?(?:동|가))\s+\d", t)
        if m:
            return m.group(1)
    return ""


def road_key(gu, addr):
    """'서울특별시 강남구 선릉로 35 (개포동)' -> '강남구|선릉로35'"""
    s = re.sub(r"\(.*?\)", "", str(addr or ""))
    s = s.replace(SIDO, "").replace("서울시", "")
    s = re.sub(r"\s+", "", s)
    s = re.sub(r"^\S*?구", "", s)                     # 앞의 자치구 제거
    m = re.match(r"(.+?(?:로|길)(?:\d+[가-힣]?길)?)(\d+(?:-\d+)?)", s)
    return f"{gu}|{m.group(1)}{m.group(2)}" if m else ""


def jibun_key(gu, addr):
    """'서울특별시 금천구 독산동 1060-18' -> '금천구|독산동1060-18'"""
    s = re.sub(r"\(.*?\)", "", str(addr or ""))
    s = s.replace(SIDO, "").replace("서울시", "")
    s = re.sub(r"\s+", "", s)
    s = re.sub(r"^\S*?구", "", s)
    m = re.match(r"(\S+?(?:동|가))(\d+(?:-\d+)?)", s)
    return f"{gu}|{m.group(1)}{m.group(2)}" if m else ""


def norm_name(s):
    """괄호, 공백, 구립/시립 같은 수식어를 떼고 비교용 이름을 만든다."""
    s = re.sub(r"\(.*?\)", "", str(s or ""))
    s = re.sub(r"[\s()（）·ㆍ,]", "", s)
    s = re.sub(r"^(구립|시립|공립|사립)", "", s)
    return s


def name_key(gu, name):
    return f"{gu}|{norm_name(name)}"

## 3. 기존 무더위쉼터

집합 K이자 좌표 공급원이다. 주소로 자치구를 붙이고 도로명, 지번, 이름 세 가지 색인을 만든다.

In [3]:
sh = pd.read_csv(P_SHELTER, encoding="cp949", dtype=str)
sh["자치구"] = [gu_of(f"{a} {b}") for a, b in zip(sh["도로명주소"], sh["지번주소"])]
sh["위도"] = pd.to_numeric(sh["위도"], errors="coerce")
sh["경도"] = pd.to_numeric(sh["경도"], errors="coerce")
sh = sh[sh["자치구"] != ""].copy()

sh["rkey"] = [road_key(g, a) for g, a in zip(sh["자치구"], sh["도로명주소"])]
sh["jkey"] = [jibun_key(g, a) for g, a in zip(sh["자치구"], sh["지번주소"])]
sh["nkey"] = [name_key(g, n) for g, n in zip(sh["자치구"], sh["쉼터명칭"])]
sh["법정동"] = [dong_of(a, b) for a, b in zip(sh["도로명주소"], sh["지번주소"])]

ok = sh["위도"].between(37.4, 37.72) & sh["경도"].between(126.7, 127.25)
print("기존 쉼터:", len(sh), "/ 좌표 정상:", ok.sum(), "/ 자치구:", sh["자치구"].nunique())

src = sh[ok].copy()
by_road = src[src["rkey"] != ""].drop_duplicates("rkey").set_index("rkey")
by_jibun = src[src["jkey"] != ""].drop_duplicates("jkey").set_index("jkey")
by_name = src.drop_duplicates("nkey").set_index("nkey")
print("색인:", len(by_road), "도로명 /", len(by_jibun), "지번 /", len(by_name), "이름")

기존 쉼터: 4092 / 좌표 정상: 4092 / 자치구: 25
색인: 3802 도로명 / 3638 지번 / 4059 이름


## 4. 후보 원자료를 공통 스키마로

금천구 파일과 같은 열을 쓴다. `시설수`는 뒤에서 같은 주소를 합칠 때 채운다.

In [4]:
COLS = ["시설유형", "시설명", "자치구", "법정동", "도로명주소", "지번주소", "위도", "경도", "좌표출처"]


def blank(n):
    return pd.Series([np.nan] * n, dtype="float64")


# 4-1. 경로당
kd = pd.read_csv(P_GYEONGRO, encoding="cp949", dtype=str)
kd = kd[kd["시도명"].fillna("").str.contains("서울")].reset_index(drop=True).copy()
gyeongro = pd.DataFrame({
    "시설유형": "경로당",
    "시설명": kd["시설명(경로당명)"].str.strip(),
    "자치구": kd["시군구명"].str.strip(),
    "법정동": [dong_of(a) for a in kd["주소(도로명)"]],
    "도로명주소": kd["주소(도로명)"].str.strip(),
    "지번주소": "",
    "위도": blank(len(kd)), "경도": blank(len(kd)), "좌표출처": "",
})

# 4-2. 주민센터
dc = pd.read_csv(P_DONG, encoding="utf-8-sig", dtype=str)
dc = dc[dc["시도"] == "서울"].reset_index(drop=True).copy()
dongsenter = pd.DataFrame({
    "시설유형": "주민센터",
    "시설명": dc["읍면동"].str.strip(),
    "자치구": dc["시군구"].str.strip(),
    "법정동": [dong_of(a) for a in dc["주소"]],
    "도로명주소": dc["주소"].str.strip(),
    "지번주소": "",
    "위도": blank(len(dc)), "경도": blank(len(dc)), "좌표출처": "",
})

# 4-3. 도서관 (좌표 있음)
lb = pd.read_csv(P_LIB, encoding="cp949", dtype=str)
lb = lb[lb["시도명"].fillna("").str.contains("서울")].reset_index(drop=True).copy()
lib = pd.DataFrame({
    "시설유형": "도서관(" + lb["도서관유형"].fillna("기타") + ")",
    "시설명": lb["도서관명"].str.strip(),
    "자치구": [gu_of(a) or g for a, g in zip(lb["소재지도로명주소"], lb["시군구명"].fillna(""))],
    "법정동": [dong_of(a) for a in lb["소재지도로명주소"]],
    "도로명주소": lb["소재지도로명주소"].str.strip(),
    "지번주소": "",
    "위도": pd.to_numeric(lb["위도"], errors="coerce"),
    "경도": pd.to_numeric(lb["경도"], errors="coerce"),
    "좌표출처": "전국도서관표준데이터",
})
lib.loc[lib["위도"].isna(), "좌표출처"] = ""

# 4-4. 보건소 (파일이 있을 때만)
parts = [gyeongro, dongsenter, lib]
if P_HEALTH is not None:
    hb = pd.read_csv(P_HEALTH, encoding="cp949", dtype=str)
    cols = list(hb.columns)

    def pick(*keys, skip=()):
        for c in cols:
            if any(k in c for k in keys) and not any(s in c for s in skip):
                return c
        return None

    c_sido, c_gu = pick("시도"), pick("시군구")
    c_name = pick("기관명", "보건기관", "명칭")
    c_addr = pick("주소", skip=("홈페이지", "누리집", "우편"))
    c_kind = pick("구분", "종류", "유형")
    # 서울만 남기면 행 번호가 흩어지므로 반드시 다시 매긴다. 안 그러면 아래에서 길이가 어긋난다.
    hb = hb[hb[c_sido].fillna("").str.contains("서울")].reset_index(drop=True).copy()
    health = pd.DataFrame({
        "시설유형": "보건소",
        "시설명": hb[c_name].str.strip(),
        "자치구": hb[c_gu].str.strip(),
        "법정동": [dong_of(a) for a in hb[c_addr]],
        "도로명주소": hb[c_addr].str.replace(r"\s+", " ", regex=True).str.strip(),
        "지번주소": "",
        "위도": blank(len(hb)), "경도": blank(len(hb)), "좌표출처": "",
    })
    parts.append(health)
    print("보건기관 서울:", len(health), "/ 구분:", hb[c_kind].value_counts().to_dict() if c_kind else "-")

cand = pd.concat(parts, ignore_index=True)[COLS]
cand = cand[cand["자치구"].str.endswith("구")].copy().reset_index(drop=True)
print("후보 원자료:", len(cand))
print(cand.groupby("시설유형").size().to_string())
print("자치구:", cand["자치구"].nunique(), "개 / 좌표 보유:", cand["위도"].notna().sum())

보건기관 서울: 63 / 구분: {'일반보건지소': 38, '보건소': 25}
후보 원자료: 4670
시설유형
경로당            3644
도서관(공공도서관)      204
도서관(어린이도서관)       4
도서관(작은도서관)      326
도서관(학교도서관)        2
보건소              63
주민센터            427
자치구: 25 개 / 좌표 보유: 535


## 5. 좌표 채우기

무더위쉼터 자료에서 도로명, 지번, 시설명 순으로 찾는다. 어느 경로로 찾았는지 `좌표출처`에 남긴다.

In [5]:
cand["rkey"] = [road_key(g, a) for g, a in zip(cand["자치구"], cand["도로명주소"])]
cand["jkey"] = [jibun_key(g, a) for g, a in zip(cand["자치구"], cand["지번주소"])]
cand["nkey"] = [name_key(g, n) for g, n in zip(cand["자치구"], cand["시설명"])]


def fill_from(idx_df, key_col, label):
    """좌표가 빈 행을 색인에서 찾아 채운다. 채운 건수를 돌려준다."""
    need = cand["위도"].isna() & (cand[key_col] != "") & cand[key_col].isin(idx_df.index)
    if not need.any():
        return 0
    keys = cand.loc[need, key_col]
    cand.loc[need, "위도"] = idx_df.loc[keys, "위도"].to_numpy()
    cand.loc[need, "경도"] = idx_df.loc[keys, "경도"].to_numpy()
    cand.loc[need, "좌표출처"] = label
    # 법정동이 비어 있으면 쉼터 자료의 법정동으로 채운다.
    got = pd.Series(idx_df.loc[keys, "법정동"].to_numpy(), index=cand.index[need])
    empty = cand.loc[need, "법정동"].fillna("") == ""
    cand.loc[got.index[empty], "법정동"] = got[empty]
    return int(need.sum())


n1 = fill_from(by_road, "rkey", "무더위쉼터(도로명주소 일치)")
n2 = fill_from(by_jibun, "jkey", "무더위쉼터(지번주소 일치)")
n3 = fill_from(by_name, "nkey", "무더위쉼터(시설명 일치)")

print(f"도로명 일치 {n1} / 지번 일치 {n2} / 이름 일치 {n3}")
print("좌표 보유:", cand["위도"].notna().sum(), "/", len(cand),
      f"({cand['위도'].notna().mean()*100:.1f}%)")

도로명 일치 2961 / 지번 일치 0 / 이름 일치 164
좌표 보유: 3660 / 4670 (78.4%)


In [6]:
miss = cand[cand["위도"].isna()]
print("좌표 없음:", len(miss), "건")
print(miss.groupby("시설유형").size().to_string())
print()
print("[자치구별 좌표 없음 상위 10]")
print(miss["자치구"].value_counts().head(10).to_string())
print()
print("[좌표출처별]")
print(cand["좌표출처"].replace("", "없음").value_counts().to_string())

좌표 없음: 1010 건
시설유형
경로당     972
보건소      37
주민센터      1

[자치구별 좌표 없음 상위 10]
자치구
강남구     149
마포구      95
서초구      78
강서구      76
동대문구     66
송파구      52
영등포구     49
광진구      42
양천구      39
은평구      39

[좌표출처별]
좌표출처
무더위쉼터(도로명주소 일치)    2961
없음                 1010
전국도서관표준데이터          535
무더위쉼터(시설명 일치)       164


## 5-2. 남은 주소 지오코딩

여기까지는 외부 호출이 없다. 남은 것만 ArcGIS 주소 지점 검색으로 찾고, 실패하면 OSM Nominatim으로 한 번 더 시도한다. 점수 100에 주소 지점(PointAddress)이고 건물번호가 일치할 때만 받는다. 응답은 `_geocode_seoul.json`에 저장하므로 다시 실행하면 호출이 나가지 않는다.

In [7]:
import json
import time

import requests

_cache = json.loads(CACHE.read_text(encoding="utf-8")) if CACHE.exists() else {}
UA = {"User-Agent": "seoul-heat-shelter-study/0.1"}


def _save():
    CACHE.write_text(json.dumps(_cache, ensure_ascii=False), encoding="utf-8")


def _get(url, params, pause):
    key = url + "|" + json.dumps(params, ensure_ascii=False, sort_keys=True)
    if key not in _cache:
        r = requests.get(url, headers=UA, params=params, timeout=30)
        r.raise_for_status()
        _cache[key] = r.json()
        _save()
        time.sleep(pause)
    return _cache[key]


def split_road(addr):
    """주소에서 도로명과 건물번호를 뽑는다.

    자료마다 표기가 달라 아래를 모두 받아야 한다.
        강동구 천호대로 1240(둔촌동, 프라자아파트)     -> 천호대로 / 1240
        서울특별시 강북구 삼양로111길 56 (수유동)       -> 삼양로111길 / 56
        서울특별시 강북구 오패산로 202(미아동)          -> 오패산로 / 202
        서울특별시 관악구 신림로29길 8, 104동 뒤편(...)  -> 신림로29길 / 8
        강남구 개포로 110길 43                          -> 개포로110길 / 43
    """
    s = str(addr or "")
    s = re.sub(r"[(（].*", "", s)          # 괄호부터 끝까지 버린다
    s = re.sub(r",.*", "", s)              # 쉼표 뒤의 동·층 표기도 버린다
    s = s.replace(SIDO, "").replace("서울시", "")
    s = re.sub(r"\s+", " ", s).strip()     # 시도를 지운 자리의 공백을 없앤다
    # 문자열 처음이 아니라 어디서든 찾는다. 도로명 안의 숫자(삼양로111길)는 도로명에 포함하고,
    # '개포로 110길'처럼 띄어 쓴 경우도 한 도로명으로 붙여 읽는다.
    m = re.search(r"([가-힣A-Za-z0-9]+(?:로|길)(?:\s*\d+[가-힣]?길)?)\s*(\d+(?:-\d+)?)", s)
    return (m.group(1).replace(" ", ""), m.group(2)) if m else (None, None)


# 자치구별 좌표 범위. 기존 쉼터 4,092곳에서 만들고 여유를 둔다.
# 지오코딩 결과가 엉뚱한 구로 찍히면 버리기 위한 검사다.
_b = src.groupby("자치구").agg(la=("위도", "min"), LA=("위도", "max"),
                              lo=("경도", "min"), LO=("경도", "max"))
GU_BBOX = {g: (r.la - 0.01, r.LA + 0.01, r.lo - 0.012, r.LO + 0.012) for g, r in _b.iterrows()}


def in_gu(gu, lat, lon):
    b = GU_BBOX.get(gu)
    return b is None or (b[0] <= lat <= b[1] and b[2] <= lon <= b[3])


def arcgis(gu, addr):
    road, num = split_road(addr)
    if not road:
        return None
    js = _get("https://geocode.arcgis.com/arcgis/rest/services/World/GeocodeServer/findAddressCandidates",
              {"SingleLine": f"{SIDO} {gu} {road} {num}", "f": "json", "countryCode": "KOR",
               "maxLocations": 3, "outFields": "Addr_type,LongLabel,Nbrhd"}, 0.3)
    for c in js.get("candidates", []):
        at = c["attributes"]
        if (c["score"] == 100 and at.get("Addr_type") == "PointAddress"
                and gu in str(at.get("LongLabel")) and str(at["LongLabel"]).rstrip().endswith(" " + num)):
            return c["location"]["y"], c["location"]["x"], "ArcGIS 지오코딩(주소 지점 일치)"
    return None


def nominatim(gu, addr):
    road, num = split_road(addr)
    if not road:
        return None
    js = _get("https://nominatim.openstreetmap.org/search",
              {"street": f"{num} {road}", "county": gu, "state": SIDO, "format": "jsonv2",
               "addressdetails": 1, "limit": 10, "countrycodes": "kr"}, 1.1)
    for x in js:
        a = x.get("address", {})
        if gu not in x.get("display_name", ""):
            continue
        if a.get("road", "").replace(" ", "") == road and a.get("house_number", "").strip(" ,") == num:
            return float(x["lat"]), float(x["lon"]), "OSM Nominatim(도로명+번호 일치)"
    return None


todo = cand.index[cand["위도"].isna()]
print("지오코딩 대상:", len(todo), "건")

found = rejected = 0
for n, i in enumerate(todo, 1):
    gu, addr = cand.at[i, "자치구"], cand.at[i, "도로명주소"]
    hit = arcgis(gu, addr) or nominatim(gu, addr)
    if hit and not in_gu(gu, hit[0], hit[1]):
        rejected += 1          # 자치구 범위를 벗어난 좌표는 버린다
        hit = None
    if hit:
        cand.at[i, "위도"], cand.at[i, "경도"], cand.at[i, "좌표출처"] = hit
        found += 1
    if n % 100 == 0 or n == len(todo):
        print(f"  {n}/{len(todo)} 처리, 찾음 {found}")

print("지오코딩으로 채움:", found, "건 / 자치구 범위를 벗어나 버린 좌표:", rejected, "건")
print("좌표 보유:", cand["위도"].notna().sum(), "/", len(cand),
      f"({cand['위도'].notna().mean()*100:.1f}%)")
print("남은 좌표 없음:", cand["위도"].isna().sum(), "건")

지오코딩 대상: 1010 건
  100/1010 처리, 찾음 96
  200/1010 처리, 찾음 196


  300/1010 처리, 찾음 296
  400/1010 처리, 찾음 389
  500/1010 처리, 찾음 487
  600/1010 처리, 찾음 585


  700/1010 처리, 찾음 685
  800/1010 처리, 찾음 783
  900/1010 처리, 찾음 882
  1000/1010 처리, 찾음 982
  1010/1010 처리, 찾음 992
지오코딩으로 채움: 992 건 / 자치구 범위를 벗어나 버린 좌표: 0 건
좌표 보유: 4652 / 4670 (99.6%)
남은 좌표 없음: 18 건


In [8]:
still = cand[cand["위도"].isna()]
print("[유형별]"); print(still.groupby("시설유형").size().to_string())
print()
print("[주소에서 도로명+번호를 못 뽑은 경우]",
      sum(1 for a in still["도로명주소"] if split_road(a)[0] is None), "건")
still[["시설유형", "시설명", "자치구", "도로명주소"]].head(10)

[유형별]
시설유형
경로당    18

[주소에서 도로명+번호를 못 뽑은 경우] 1 건


,시설유형,시설명,자치구,도로명주소
31,경로당,웃말,강동구,강동구 천호대로162길 72(성내동)
105,경로당,명일1동,강동구,강동구 양재대로138길 29(명일동)
266,경로당,몽촌,송파구,송파구 백제고분로 489 (방이동)
322,경로당,역삼복지경로당,강남구,강남구 언주로70길 18
807,경로당,사당솔밭,동작구,사당로 189-9
826,경로당,학수,동작구,사당로27길 215
930,경로당,신길5동시범길,영등포구,도림로63길 37-5(신길5동 405-43)
1005,경로당,양평3가,영등포구,선유동2로 7-11(양평동3가 77-65)
1023,경로당,문래진주(아),영등포구,선유로9길 31(문래동5가 22)
1043,경로당,일심,영등포구,문래북로 128(문래동3가 31)


## 5-3. 남은 곳은 지번으로 한 번 더

도로명으로 못 찾은 주소 중에는 괄호 안에 지번이 적힌 것이 있다. `선유동2로 7-11(양평동3가 77-65)`처럼이다. 지번으로 다시 조회한다. 지번 검색은 도로명보다 결과가 느슨하므로 자치구 범위 검사를 반드시 통과해야 받는다.

In [9]:
def split_jibun(addr):
    """'선유동2로 7-11(양평동3가 77-65)' -> ('양평동3가', '77-65')"""
    m = re.search(r"[(（]([^)）]*)[)）]", str(addr or ""))
    inner = m.group(1) if m else str(addr or "")
    m2 = re.search(r"([가-힣0-9]+(?:동|가))\s*(\d+(?:-\d+)?)", inner)
    return (m2.group(1), m2.group(2)) if m2 else (None, None)


def arcgis_jibun(gu, addr):
    dong, num = split_jibun(addr)
    if not dong:
        return None
    js = _get("https://geocode.arcgis.com/arcgis/rest/services/World/GeocodeServer/findAddressCandidates",
              {"SingleLine": f"{SIDO} {gu} {dong} {num}", "f": "json", "countryCode": "KOR",
               "maxLocations": 3, "outFields": "Addr_type,LongLabel,Nbrhd"}, 0.3)
    for c in js.get("candidates", []):
        at = c["attributes"]
        label = str(at.get("LongLabel"))
        if c["score"] >= 95 and gu in label and dong[:2] in label:
            return c["location"]["y"], c["location"]["x"], "ArcGIS 지오코딩(지번 일치)"
    return None


todo2 = cand.index[cand["위도"].isna()]
found2 = 0
for i in todo2:
    gu, addr = cand.at[i, "자치구"], cand.at[i, "도로명주소"]
    hit = arcgis_jibun(gu, addr)
    if hit and in_gu(gu, hit[0], hit[1]):
        cand.at[i, "위도"], cand.at[i, "경도"], cand.at[i, "좌표출처"] = hit
        found2 += 1

print("지번으로 추가로 채움:", found2, "/", len(todo2), "건")
print("최종 좌표 보유:", cand["위도"].notna().sum(), "/", len(cand),
      f"({cand['위도'].notna().mean()*100:.1f}%)")
print("최종 좌표 없음:", cand["위도"].isna().sum(), "건")
cand[cand["위도"].isna()][["시설유형", "시설명", "자치구", "도로명주소"]]

지번으로 추가로 채움: 4 / 18 건
최종 좌표 보유: 4656 / 4670 (99.7%)
최종 좌표 없음: 14 건


,시설유형,시설명,자치구,도로명주소
31,경로당,웃말,강동구,강동구 천호대로162길 72(성내동)
105,경로당,명일1동,강동구,강동구 양재대로138길 29(명일동)
266,경로당,몽촌,송파구,송파구 백제고분로 489 (방이동)
322,경로당,역삼복지경로당,강남구,강남구 언주로70길 18
807,경로당,사당솔밭,동작구,사당로 189-9
826,경로당,학수,동작구,사당로27길 215
930,경로당,신길5동시범길,영등포구,도림로63길 37-5(신길5동 405-43)
1297,경로당,호국 보훈 구로 경로당,구로구,구로구 우마1다길 25 (가리봉동)
1500,경로당,내발산,강서구,강서구 우장산로 24-7
1641,경로당,청솔,양천구,중앙로52길 73(신정동)


## 6. 같은 주소의 시설 합치기

한 건물에 경로당이 여럿 있으면 입지로는 한 지점이다. 도로명주소 키가 같으면 한 행으로 합치고 합친 수를 `시설수`에 남긴다.

In [10]:
cand["_k"] = np.where(cand["rkey"] != "", cand["rkey"],
                      np.where(cand["jkey"] != "", cand["jkey"],
                               cand["자치구"] + "|" + cand["시설명"]))


def merge_group(g):
    lat = g["위도"].dropna()
    lon = g["경도"].dropna()
    src_ = [s for s in g["좌표출처"] if s]
    dong = [d for d in g["법정동"] if d]
    return pd.Series({
        "시설유형": "/".join(dict.fromkeys(g["시설유형"])),
        "시설명": ", ".join(dict.fromkeys(g["시설명"])),
        "시설수": len(g),
        "자치구": g["자치구"].iloc[0],
        "법정동": dong[0] if dong else "",
        "상세주소": g["도로명주소"].iloc[0],
        "위도": lat.iloc[0] if len(lat) else np.nan,
        "경도": lon.iloc[0] if len(lon) else np.nan,
        "좌표출처": src_[0] if src_ else "",
        "rkey": g["rkey"].iloc[0], "jkey": g["jkey"].iloc[0], "nkey": g["nkey"].iloc[0],
    })


merged = cand.groupby("_k", sort=False).apply(merge_group, include_groups=False).reset_index(drop=True)
merged["시구동"] = (SIDO + " " + merged["자치구"] + " " + merged["법정동"]).str.strip()
print(f"{len(cand)}행 -> {len(merged)}행 (합쳐진 시설 {len(cand) - len(merged)}개)")
print("시설수 2 이상:", (merged["시설수"] > 1).sum(), "곳")
print("법정동 없음:", (merged["법정동"] == "").sum(), "곳")

4670행 -> 4310행 (합쳐진 시설 360개)
시설수 2 이상: 319 곳
법정동 없음: 753 곳


## 7. 기존 무더위쉼터와의 교집합

도로명주소, 지번주소, 시설명 셋 중 하나라도 같으면 이미 지정된 것으로 본다. 무엇으로 맞췄는지 `일치기준`에 남긴다.

In [11]:
def find_existing(r):
    for how, key, idx in (("도로명주소", r["rkey"], by_road),
                          ("지번주소", r["jkey"], by_jibun),
                          ("시설명", r["nkey"], by_name)):
        if key and key in idx.index:
            h = idx.loc[key]
            return pd.Series({"일치기준": how, "기존쉼터명칭": h["쉼터명칭"],
                              "기존쉼터_도로명주소": h["도로명주소"]})
    return pd.Series({"일치기준": "", "기존쉼터명칭": "", "기존쉼터_도로명주소": ""})


hit = merged.apply(find_existing, axis=1)
full = pd.concat([merged, hit], axis=1)

OUT_COLS = ["시설유형", "시설명", "시구동", "시설수", "상세주소", "위도", "경도", "좌표출처"]
inter = full[full["일치기준"] != ""]
rest = full[full["일치기준"] == ""]

print("후보 전체:", len(full), "/ 교집합:", len(inter), "/ 신규 후보:", len(rest))
print()
print("[일치기준]"); print(inter["일치기준"].value_counts().to_string())
print()
print("[신규 후보 시설유형]"); print(rest["시설유형"].value_counts().to_string())

후보 전체: 4310 / 교집합: 3149 / 신규 후보: 1161

[일치기준]
일치기준
도로명주소    2971
시설명       178

[신규 후보 시설유형]
시설유형
경로당                       943
도서관(작은도서관)                119
도서관(공공도서관)                 48
보건소                        37
경로당/도서관(작은도서관)              7
도서관(어린이도서관)                 3
도서관(학교도서관)                  2
주민센터                        1
도서관(작은도서관)/도서관(어린이도서관)      1


## 8. 저장

In [12]:
def save(df, name, extra=()):
    d = df[OUT_COLS + list(extra)].copy()
    for c in ("위도", "경도"):
        d[c] = d[c].map(lambda v: "" if pd.isna(v) else round(float(v), 7))
    d = d.sort_values(["시구동", "시설유형", "시설명"])
    d.to_csv(PROC / name, index=False, encoding="utf-8-sig")
    print(name, len(d), "행")
    return d


save(full, "서울시_후보시설.csv")
save(inter, "서울시_후보시설_교집합.csv", ("일치기준", "기존쉼터명칭", "기존쉼터_도로명주소"))
save(rest, "서울시_후보시설_기존쉼터제외.csv")

print()
print("[자치구별 신규 후보 상위 10]")
print(rest["자치구"].value_counts().head(10).to_string())
print()
print("[신규 후보 중 좌표 없음]", rest["위도"].isna().sum(), "건")
rest.head(10)[OUT_COLS]

서울시_후보시설.csv 4310 행


서울시_후보시설_교집합.csv 3149 행
서울시_후보시설_기존쉼터제외.csv 1161 행

[자치구별 신규 후보 상위 10]


자치구
강남구     152
마포구      95
은평구      83
강서구      82
서초구      79
동대문구     64
광진구      57
영등포구     54
송파구      53
금천구      45

[신규 후보 중 좌표 없음] 14 건


,시설유형,시설명,시구동,시설수,상세주소,위도,경도,좌표출처
10,경로당,둔촌프라자A,서울특별시 강동구 둔촌동,1,"강동구 천호대로 1240(둔촌동, 프라자아파트)",37.535403,127.148067,ArcGIS 지오코딩(주소 지점 일치)
15,경로당,길동삼익파크A(1),서울특별시 강동구 길동,1,"강동구 명일로 286(길동, 삼익파크아파트)",37.544350,127.148513,ArcGIS 지오코딩(주소 지점 일치)
17,경로당,한빛(구 골드빌),서울특별시 강동구 길동,1,"강동구 성안로 176(길동, 길동골드빌1차아파트)",37.538508,127.136364,ArcGIS 지오코딩(주소 지점 일치)
19,경로당,길동우성A,서울특별시 강동구 길동,1,"강동구 천호대로193길 37(길동, 길동우성아파트)",37.537874,127.143541,ArcGIS 지오코딩(주소 지점 일치)
23,경로당,달님,서울특별시 강동구 길동,1,강동구 양재대로112길 52(길동),37.536210,127.142568,ArcGIS 지오코딩(주소 지점 일치)
25,경로당,둔촌역청구,서울특별시 강동구 성내동,1,"강동구 풍성로 194(성내동, 둔촌역청구아파트)",37.529601,127.133361,ArcGIS 지오코딩(주소 지점 일치)
29,경로당,성내현대A,서울특별시 강동구 성내동,1,"강동구 천호옛12길27(성내동, 현대아파트)",37.535515,127.127031,ArcGIS 지오코딩(주소 지점 일치)
31,경로당,웃말,서울특별시 강동구 성내동,1,강동구 천호대로162길 72(성내동),NaN,NaN,
35,경로당,성안마을청구A,서울특별시 강동구 성내동,1,"강동구 풍성로 116(성내동, 성안마을청구아파트)",37.531889,127.124961,ArcGIS 지오코딩(주소 지점 일치)
36,경로당,성내삼성A,서울특별시 강동구 성내동,1,"강동구 풍성로 128(성내동, 삼성2단지아파트)",37.531258,127.126282,ArcGIS 지오코딩(주소 지점 일치)


## 9. 다음 단계

좌표가 빈 후보가 남는다. 대량 지오코딩은 서비스 약관과 호출 한도 문제가 있으므로, 남은 건수를 보고 방법을 정한다. 행정안전부 도로명주소 건물 데이터베이스를 내려받아 로컬에서 주소를 맞추면 외부 호출 없이 채울 수 있고, 인터넷이 없는 데이터안심구역에서도 같은 코드가 돈다.

보건소, 복지관, 체육시설, 은행은 자료를 확보하는 대로 4절에 원천을 추가하면 된다. 이후 절차는 그대로 쓸 수 있다.